# 04. Поведенческая сегментация

> **Краткое резюме**: Кластеризуем клиентов по поведенческому профилю (обороты, контрагенты, направленность, динамика). Пересечение с модельным сегментом показывает, насколько поведение совпадает с формальной классификацией. Результат — каждый клиент получает `behavioral_segment`.

**Предпосылки**: `03_feature_engineering.ipynb` выполнен.

---

In [ ]:
import os
import pickle

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import calinski_harabasz_score, davies_bouldin_score, silhouette_score
from sklearn.preprocessing import MinMaxScaler

OUTPUT_DIR = os.path.abspath("./data")

In [ ]:
# =====================================================
# ПАРАМЕТРЫ
# =====================================================

K_MIN = 3  # Минимальное число кластеров
K_MAX = 8  # Максимальное
# None = автоподбор по silhouette + Calinski-Harabasz
# Число = фиксированный k (для debug / быстрого прогона)
K_OVERRIDE = None

# Порог обрезки выбросов в нормализованном пространстве (±σ)
# Предотвращает формирование кластеров из 1–2 клиентов
CLIP_SIGMA = 3.0

---
## 1. Загрузка данных

In [ ]:
X = pd.read_parquet(os.path.join(OUTPUT_DIR, "feature_matrix.parquet"))
full_df = pd.read_parquet(os.path.join(OUTPUT_DIR, "full_client_data.parquet"))

with open(os.path.join(OUTPUT_DIR, "feature_meta.pkl"), "rb") as f:
    meta = pickle.load(f)

print(f"Feature matrix: {X.shape}")
print(f"Full data: {full_df.shape}")

---
## 1.5. Винзоризация выбросов

Клиенты-хабы с аномально высоким числом транзакций или контрагентов (например, 3000+ tx, 2500+ counterparties)
формируют отдельный кластер из 1 человека — это артефакт, а не бизнес-сегмент.

Обрезаем значения за пределами ±{CLIP_SIGMA}σ в нормализованном пространстве.
Клиенты **остаются** в данных, просто их экстремальные координаты приводятся к разумному диапазону.

In [ ]:
# Обрезаем значения в нормализованном пространстве
X_clipped = X.clip(lower=-CLIP_SIGMA, upper=CLIP_SIGMA)

n_outlier_vals = (X.abs() > CLIP_SIGMA).sum().sum()
n_outlier_clients = (X.abs() > CLIP_SIGMA).any(axis=1).sum()
print(f"Clipped {n_outlier_vals:,} values exceeding ±{CLIP_SIGMA}σ")
print(f"Affected clients: {n_outlier_clients:,} ({n_outlier_clients / len(X) * 100:.1f}%)")

# Показываем топ-5 клиентов по «экстремальности» (только числовые столбцы)
extreme_score = X.select_dtypes(include="number").abs().max(axis=1)
print("\nТоп-5 самых экстремальных клиентов (до обрезки):")
print(extreme_score.nlargest(5))

In [ ]:
X_vals = X_clipped.values  # используем обрезанные данные

if K_OVERRIDE is not None:
    best_k = K_OVERRIDE
    print(f"Using override k={best_k}")
    res_df = None
else:
    max_k = min(K_MAX, len(X) - 1)
    results = []
    for k in range(K_MIN, max_k + 1):
        km = KMeans(n_clusters=k, random_state=42, n_init=10)
        labels = km.fit_predict(X_vals)
        if len(set(labels)) < 2:
            continue
        sil = silhouette_score(X_vals, labels, sample_size=min(10_000, len(X)))
        ch = calinski_harabasz_score(X_vals, labels)
        db = davies_bouldin_score(X_vals, labels)
        results.append(
            {
                "k": k,
                "silhouette": sil,
                "calinski_harabasz": ch,
                "davies_bouldin": db,
                "inertia": km.inertia_,
            }
        )
        print(f"  k={k}: silhouette={sil:.4f}  CH={ch:.1f}  DB={db:.4f}")

    res_df = pd.DataFrame(results)

    def minmax(s: pd.Series) -> pd.Series:
        rng = s.max() - s.min()
        return (s - s.min()) / rng if rng > 0 else pd.Series(0.5, index=s.index)

    res_df["score"] = (
        minmax(res_df["silhouette"])
        + minmax(res_df["calinski_harabasz"])
        - minmax(res_df["davies_bouldin"])
    )
    best_k = int(res_df.loc[res_df["score"].idxmax(), "k"])
    best_row = res_df.loc[res_df["score"].idxmax()]
    print(
        f"\nBest k={best_k}  (silhouette={best_row['silhouette']:.4f},"
        f"  CH={best_row['calinski_harabasz']:.1f},"
        f"  DB={best_row['davies_bouldin']:.4f})"
    )
    display(res_df.round(4))

In [ ]:
if res_df is not None and len(res_df) > 1:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    axes[0, 0].plot(res_df["k"], res_df["silhouette"], "o-", linewidth=2)
    axes[0, 0].axvline(best_k, color="red", linestyle="--", label=f"best k={best_k}")
    axes[0, 0].set_title("Silhouette Score (↑ лучше)")
    axes[0, 0].legend()

    axes[0, 1].plot(res_df["k"], res_df["inertia"], "o-", linewidth=2, color="orange")
    axes[0, 1].axvline(best_k, color="red", linestyle="--")
    axes[0, 1].set_title("Inertia / Elbow (ищем «локоть»)")

    axes[1, 0].plot(res_df["k"], res_df["calinski_harabasz"], "o-", linewidth=2, color="green")
    axes[1, 0].axvline(best_k, color="red", linestyle="--")
    axes[1, 0].set_title("Calinski-Harabasz (↑ лучше)")

    axes[1, 1].plot(res_df["k"], res_df["davies_bouldin"], "o-", linewidth=2, color="purple")
    axes[1, 1].axvline(best_k, color="red", linestyle="--")
    axes[1, 1].set_title("Davies-Bouldin (↓ лучше)")

    for ax in axes.flat:
        ax.set_xlabel("k")
    plt.suptitle("Выбор оптимального k кластеров", fontsize=14, y=1.01)
    plt.tight_layout()
    plt.show()

---
## 3. Финальная кластеризация

In [ ]:
km_final = KMeans(n_clusters=best_k, random_state=42, n_init=10)
full_df["behavioral_segment"] = km_final.fit_predict(X_vals)

print(f"Segments assigned (k={best_k}):")
print(full_df["behavioral_segment"].value_counts().sort_index())

---
## 4. Профили сегментов

Средние значения ключевых поведенческих метрик по сегментам.

In [ ]:
PROFILE_COLS = [
    "total_amount",
    "tx_count",
    "unique_counterparties",
    "direction_ratio",
    "active_months",
    "amount_growth",
    "cp_growth",
]
for col in ["avg_balance", "product_count", "avg_monthly_amount", "tx_amount_cv"]:
    if col in full_df.columns:
        PROFILE_COLS.append(col)

segment_profiles = full_df.groupby("behavioral_segment")[PROFILE_COLS].mean()
print("Segment profiles (mean):")
display(segment_profiles.round(2))

In [ ]:
# ruff: noqa: F821
# Пороги на основе реального распределения всей популяции, а не медианы сегментных средних.
# Это исключает ошибку когда "медианный" сегмент попадает в "Малоактивный" только потому,
# что его среднее совпадает с медианой 5 сегментных средних.
_label_cols = [
    c
    for c in ["tx_count", "unique_counterparties", "total_amount", "active_months", "amount_growth"]
    if c in full_df.columns
]
_q75 = full_df[_label_cols].quantile(0.75)  # порог «высокий»
_q95 = full_df[_label_cols].quantile(0.95)  # порог «очень высокий» (мега)
_med = full_df[_label_cols].median()


def auto_label_segment(profile: pd.Series) -> str:
    """Присваивает бизнес-название сегменту на основе популяционных квантилей."""
    tx = profile.get("tx_count", 0)
    cp = profile.get("unique_counterparties", 0)
    amt = profile.get("total_amount", 0)
    act = profile.get("active_months", 0)
    grow = profile.get("amount_growth", 0)
    dr = profile.get("direction_ratio", 0.5)

    is_mega_tx = tx >= _q95.get("tx_count", 1e9)
    is_mega_cp = cp >= _q95.get("unique_counterparties", 1e9)
    is_high_tx = tx >= _q75.get("tx_count", 1e9)
    is_high_cp = cp >= _q75.get("unique_counterparties", 1e9)
    is_growing = grow >= _q75.get("amount_growth", 1e9)
    is_high_amt = amt >= _q75.get("total_amount", 1e9)
    is_active = act >= _med.get("active_months", 0)

    if is_mega_tx and is_mega_cp:
        return "🌐 Мега-хаб (системообразующий)"
    if is_high_tx and is_high_cp:
        return "🔗 Хаб (много контрагентов, высокая активность)"
    if is_growing:
        return "🚀 Растущий (агрессивный рост оборота)"
    if dr >= 0.70:
        return "💸 Плательщик (преимущественно расходы)"
    if dr <= 0.30:
        return "📥 Получатель (преимущественно входящие)"
    if is_high_amt and is_active:
        return "⭐ Крупный активный"
    return "😴 Малоактивный / низкий оборот"


segment_labels = {seg_id: auto_label_segment(row) for seg_id, row in segment_profiles.iterrows()}
full_df["segment_label"] = full_df["behavioral_segment"].map(segment_labels)

print("Авто-маркировка сегментов:")
for seg_id, label in sorted(segment_labels.items()):
    n = (full_df["behavioral_segment"] == seg_id).sum()
    print(f"  Сегмент {seg_id} ({n:,} клиентов): {label}")

In [ ]:
norm_profiles = pd.DataFrame(
    MinMaxScaler().fit_transform(segment_profiles),
    index=segment_profiles.index,
    columns=segment_profiles.columns,
)

labels = [c[:14] for c in norm_profiles.columns]
angles = np.linspace(0, 2 * np.pi, len(labels), endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(polar=True))
colors = plt.cm.tab10.colors

for i, (seg_id, row) in enumerate(norm_profiles.iterrows()):
    vals = row.tolist() + [row.tolist()[0]]
    lbl = f"Seg {seg_id}: {segment_labels.get(seg_id, '')}"
    ax.plot(angles, vals, "o-", label=lbl, color=colors[i % len(colors)], linewidth=2)
    ax.fill(angles, vals, alpha=0.08, color=colors[i % len(colors)])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(labels, fontsize=9)
ax.set_title("Профили сегментов (нормализовано)", fontsize=13, pad=20)
ax.legend(loc="upper right", bbox_to_anchor=(1.55, 1.1), fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# ruff: noqa: F821
# ── UMAP: 2D визуализация кластеров ─────────────────────────────────────────
# UMAP сохраняет локальную структуру данных лучше PCA, особенно для
# поведенческих признаков с нелинейными зависимостями.
#
# Ошибка "AttributeError: 'float' object has no attribute 'shape'" в numpy.average
# — баг совместимости umap-learn < 0.5.6 с numpy >= 2.0.
# Лечится: uv pip install "umap-learn>=0.5.6"
try:
    import warnings

    import umap

    print("Fitting UMAP (cosine, 2D)…")
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        reducer = umap.UMAP(
            n_components=2,
            metric="cosine",
            n_neighbors=30,
            min_dist=0.1,
            random_state=42,
        )
        X_2d = reducer.fit_transform(X_vals)

    seg_ids = sorted(full_df["behavioral_segment"].unique())
    seg_assignment = full_df.loc[X.index, "behavioral_segment"].values
    colors = plt.cm.tab10.colors

    fig, ax = plt.subplots(figsize=(13, 8))
    for i, seg_id in enumerate(seg_ids):
        mask = seg_assignment == seg_id
        lbl = f"Seg {seg_id}: {segment_labels.get(seg_id, '')[:30]}"
        ax.scatter(
            X_2d[mask, 0],
            X_2d[mask, 1],
            c=[colors[i % len(colors)]],
            label=lbl,
            alpha=0.4,
            s=6,
            rasterized=True,
        )
        # Centroid marker
        cx, cy = X_2d[mask, 0].mean(), X_2d[mask, 1].mean()
        ax.scatter(
            cx, cy, c=[colors[i % len(colors)]], s=200, marker="*", edgecolors="black", zorder=5
        )
        ax.annotate(
            f"Seg {seg_id}",
            (cx, cy),
            fontsize=9,
            fontweight="bold",
            textcoords="offset points",
            xytext=(6, 6),
        )

    ax.set_title("UMAP — клиенты в 2D (cosine distance)\n★ = центроид сегмента", fontsize=13)
    ax.set_xlabel("UMAP-1")
    ax.set_ylabel("UMAP-2")
    ax.legend(loc="upper right", fontsize=8, framealpha=0.9)
    plt.tight_layout()
    plt.show()
    print("✅ UMAP: хорошо разделённые кластеры означают устойчивую сегментацию")

except ImportError:
    print("⚠️  umap-learn не установлен: uv pip install umap-learn")
except AttributeError as e:
    if "shape" in str(e):
        print("⚠️  Несовместимость umap-learn с текущей версией numpy.")
        print("   Исправление: uv pip install 'umap-learn>=0.5.6'")
        print(f"   Подробнее: {e}")
    else:
        raise

In [ ]:
# ruff: noqa: F821
# ── HDBSCAN vs KMeans: сравнение методов кластеризации ──────────────────────
# HDBSCAN не требует задавать k, автоматически определяет число кластеров
# и помечает выбросы (-1). Полезен для обнаружения аномальных клиентов.
# KMeans: каждому клиенту гарантированно присваивается сегмент.
# Рекомендация: KMeans для сегментации, HDBSCAN для выявления выбросов.
try:
    import hdbscan
    from sklearn.metrics import calinski_harabasz_score, davies_bouldin_score

    print("Fitting HDBSCAN…")
    hdb = hdbscan.HDBSCAN(
        min_cluster_size=50, min_samples=10, metric="euclidean", core_dist_n_jobs=-1
    )
    hdb_labels = hdb.fit_predict(X_vals)

    n_hdb = len(set(hdb_labels)) - (1 if -1 in hdb_labels else 0)
    n_noise = int((hdb_labels == -1).sum())
    valid = hdb_labels != -1

    sil_km = silhouette_score(X_vals, km_final.labels_, sample_size=min(10_000, len(X_vals)))
    sil_hdb = (
        silhouette_score(X_vals[valid], hdb_labels[valid], sample_size=min(10_000, valid.sum()))
        if valid.sum() > 1 and len(set(hdb_labels[valid])) > 1
        else float("nan")
    )

    ch_km = calinski_harabasz_score(X_vals, km_final.labels_)
    ch_hdb = (
        calinski_harabasz_score(X_vals[valid], hdb_labels[valid])
        if valid.sum() > 1 and len(set(hdb_labels[valid])) > 1
        else float("nan")
    )

    db_km = davies_bouldin_score(X_vals, km_final.labels_)
    db_hdb = (
        davies_bouldin_score(X_vals[valid], hdb_labels[valid])
        if valid.sum() > 1 and len(set(hdb_labels[valid])) > 1
        else float("nan")
    )

    comp = pd.DataFrame(
        {
            "Метрика": [
                "Кластеров",
                "Выбросов (шум)",
                "Silhouette ↑",
                "Calinski-Harabasz ↑",
                "Davies-Bouldin ↓",
                "Требует задать k",
                "Мягкие границы",
                "Масштаб на prod",
            ],
            f"KMeans (k={best_k})": [
                best_k,
                0,
                f"{sil_km:.4f}",
                f"{ch_km:.1f}",
                f"{db_km:.4f}",
                "Да",
                "Нет",
                "✅ O(n·k·iter)",
            ],
            "HDBSCAN": [
                n_hdb,
                n_noise,
                f"{sil_hdb:.4f}" if not pd.isna(sil_hdb) else "—",
                f"{ch_hdb:.1f}" if not pd.isna(ch_hdb) else "—",
                f"{db_hdb:.4f}" if not pd.isna(db_hdb) else "—",
                "Нет",
                "Да",
                "⚠️  O(n²) память",
            ],
        }
    ).set_index("Метрика")
    print("\n📊 СРАВНЕНИЕ МЕТОДОВ КЛАСТЕРИЗАЦИИ:")
    display(comp)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    km_counts = pd.Series(km_final.labels_).value_counts().sort_index()
    hdb_counts = pd.Series(hdb_labels).value_counts().sort_index()

    axes[0].bar(
        km_counts.index.astype(str),
        km_counts.values,
        edgecolor="black",
        alpha=0.75,
        color="#1f77b4",
    )
    axes[0].set_title(f"KMeans: {best_k} сегментов, 0 выбросов")
    axes[0].set_xlabel("Сегмент")
    axes[0].set_ylabel("Клиентов")

    bar_labels = ["шум" if i == -1 else str(i) for i in hdb_counts.index]
    bar_colors = ["#d62728" if i == -1 else "#2ca02c" for i in hdb_counts.index]
    axes[1].bar(bar_labels, hdb_counts.values, color=bar_colors, edgecolor="black", alpha=0.75)
    axes[1].set_title(f"HDBSCAN: {n_hdb} кластеров + {n_noise} выбросов (красный)")
    axes[1].set_xlabel("Кластер")
    axes[1].set_ylabel("Клиентов")

    plt.suptitle("KMeans vs HDBSCAN: распределение клиентов", fontsize=13)
    plt.tight_layout()
    plt.show()

    winner = "HDBSCAN" if (not pd.isna(sil_hdb) and sil_hdb > sil_km) else f"KMeans (k={best_k})"
    print("\n📌 ВЫВОД:")
    print(f"  • По silhouette лучше: {winner}")
    print(f"  • HDBSCAN выявил {n_noise} клиентов-аномалий ({n_noise / len(X_vals) * 100:.1f}%)")
    print("    — кандидаты для исключения из look-alike эталона")
    print("  • Для production-сегментации рекомендуется KMeans:")
    print("    каждый клиент получает сегмент, O(n) масштабируемость")
    print("  • HDBSCAN — дополнительный фильтр выбросов перед обучением GBM")

except ImportError:
    print("⚠️  hdbscan не установлен: pip install hdbscan")

In [ ]:
# ruff: noqa: F821
print("=" * 70)
print("ИНТЕРПРЕТАЦИЯ СЕГМЕНТОВ")
print("=" * 70)

KEY_METRICS = [
    "total_amount",
    "tx_count",
    "unique_counterparties",
    "direction_ratio",
    "active_months",
    "amount_growth",
]
key_cols = [c for c in KEY_METRICS if c in segment_profiles.columns]
prof = segment_profiles[key_cols].copy()
overall_med = full_df[key_cols].median()


def _describe_special(val: float, col: str) -> str | None:
    """Возвращает описание для специальных колонок (direction_ratio, amount_growth)."""
    if col == "direction_ratio":
        if val > 0.65:
            return f"▲ {val:.2f} — преимущественно расходы"
        return f"▼ {val:.2f} — входящие" if val < 0.35 else f"→ {val:.2f} — смешанный поток"
    if col == "amount_growth":
        if val > 0.3:
            return f"↑ +{val:.2f} — оборот растёт"
        return f"↓ {val:.2f} — оборот падает" if val < -0.1 else f"→ {val:.2f} — стабильный"
    return None


def describe_vs_median(val: float, med: float, col: str) -> str:
    """Описывает значение метрики относительно медианы."""
    special = _describe_special(val, col)
    if special is not None:
        return special
    ratio = val / med if med != 0 else 1.0
    if ratio > 2.0:
        return f"▲▲ {val:,.1f} (×{ratio:.1f} медианы)"
    if ratio > 1.3:
        return f"▲ {val:,.1f} (выше медианы)"
    if ratio < 0.5:
        return f"▼▼ {val:,.1f} (×{ratio:.1f} медианы)"
    if ratio < 0.8:
        return f"▼ {val:,.1f} (ниже медианы)"
    return f"→ {val:,.1f} (≈ медиана)"


for seg_id in sorted(segment_labels):
    n = (full_df["behavioral_segment"] == seg_id).sum()
    pct = n / len(full_df) * 100
    print(f"\n{'─' * 60}")
    print(f"  Сегмент {seg_id} | {segment_labels[seg_id]}")
    print(f"  Размер: {n:,} клиентов ({pct:.1f}%)")
    print("  Ключевые метрики:")
    for col in key_cols:
        desc = describe_vs_median(prof.loc[seg_id, col], overall_med[col], col)
        print(f"    {col:25s}: {desc}")

print(f"\n{'=' * 70}")
print("ВЫВОД: Для look-alike наиболее ценны клиенты из сегментов с высоким")
print("оборотом и ростом. Малоактивные сегменты — плохие кандидаты для эталона.")
print("=" * 70)

---
## 5. Пересечение с модельным сегментом

Насколько поведенческая сегментация совпадает с формальным модельным сегментом?

In [ ]:
seg_path = os.path.join(OUTPUT_DIR, "segments.parquet")
full_df.to_parquet(seg_path)
print(f"Segments saved: {seg_path}")

# Сохраняем модель KMeans + маппинг меток
model_path = os.path.join(OUTPUT_DIR, "kmeans_model.pkl")
with open(model_path, "wb") as f:
    pickle.dump({"model": km_final, "segment_labels": segment_labels}, f)
print(f"KMeans model + labels saved: {model_path}")

In [ ]:
# Пересечение с ОКВЭД
if "sparkokved_ccode" in full_df.columns:
    okved_cross = pd.crosstab(
        full_df["sparkokved_ccode"].fillna("N/A"),
        full_df["behavioral_segment"],
        margins=True,
        margins_name="Total",
    )
    print("Top-15 OKVED x Behavioral Segment:")
    top_okveds = okved_cross.drop("Total")["Total"].nlargest(15).index
    display(okved_cross.loc[list(top_okveds) + ["Total"]])

---
## 6. Сохранение